# stirs — Stage 2 (EffectErase) Kaggle Worker
In the right-hand **Notebook** panel, set:
- **Accelerator:** `GPU T4 x2`
- **Internet:** `On`

In [ ]:
%%bash
if [ ! -d /kaggle/working/EffectErase ]; then
  git clone https://github.com/FudanCVL/EffectErase.git /kaggle/working/EffectErase
fi
sed -i 's/weights_only=True/weights_only=False/g' /kaggle/working/EffectErase/diffsynth/models/utils.py
sed -i 's/ModelManager(device="cuda"/ModelManager(device="cpu"/g' /kaggle/working/EffectErase/examples/remove_wan/infer_remove_wan.py
sed -i 's/torch\.bfloat16/torch.float16/g' /kaggle/working/EffectErase/examples/remove_wan/infer_remove_wan.py
sed -i "s/# CLIP/# CLIP\\n        self.load_models_to_device(['image_encoder'])/" /kaggle/working/EffectErase/diffsynth/pipelines/wan_video.py
pip install -q -r /kaggle/working/EffectErase/requirements.txt fastapi uvicorn python-multipart huggingface_hub
pip install -q --no-deps -e /kaggle/working/EffectErase
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
echo "Cell 1 complete!"

In [ ]:
%%bash
mkdir -p /kaggle/working/weights
huggingface-cli download alibaba-pai/Wan2.1-Fun-1.3B-InP --local-dir /kaggle/working/weights
echo "Weights ready! Ensure EffectErase.ckpt is also in /kaggle/working/weights/EffectErase.ckpt"

In [ ]:
%%writefile /kaggle/working/kaggle_effect_erase_server.py
import os
import shutil
import subprocess
import tempfile
import threading
import uuid
from pathlib import Path

import uvicorn
from fastapi import FastAPI, UploadFile
from fastapi.responses import JSONResponse, Response

WEIGHTS_ROOT = Path(os.environ.get("EFFECT_ERASE_WEIGHTS_DIR", "/kaggle/working/weights"))
SCRIPT_PATH = Path("/kaggle/working/EffectErase/examples/remove_wan/infer_remove_wan.py")

app = FastAPI(title="stirs EffectErase Kaggle Worker")
JOBS = {}

def _run_job(job_id: str, cmd: list[str], out_path: Path, tmpdir: str):
    proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
    if proc.returncode != 0 or not out_path.exists():
        JOBS[job_id] = {"status": "failed", "error": proc.stderr[-2000:]}
    else:
        JOBS[job_id] = {"status": "done", "data": out_path.read_bytes()}
    shutil.rmtree(tmpdir, ignore_errors=True)

@app.post("/")
async def remove_object(
    fg_bg_video: UploadFile,
    mask_video: UploadFile,
    num_inference_steps: int = 25,
    height: int = 480,
    width: int = 832,
    num_frames: int = 81,
    frame_interval: int = 1,
    seed: int = 2025,
    cfg: float = 1.0,
    lora_alpha: float = 1.0,
    tiled: bool = True,
    use_teacache: bool = False,
):
    job_id = str(uuid.uuid4())
    tmpdir = tempfile.mkdtemp()
    tmp = Path(tmpdir)
    fg_path = tmp / "fg_bg.mp4"
    mask_path = tmp / "mask.mp4"
    out_path = tmp / "output.mp4"

    fg_path.write_bytes(await fg_bg_video.read())
    mask_path.write_bytes(await mask_video.read())

    cmd = [
        "python", str(SCRIPT_PATH),
        "--fg_bg_path", str(fg_path),
        "--mask_path", str(mask_path),
        "--output_path", str(out_path),
        "--text_encoder_path", str(WEIGHTS_ROOT / "models_t5_umt5-xxl-enc-bf16.pth"),
        "--vae_path", str(WEIGHTS_ROOT / "Wan2.1_VAE.pth"),
        "--dit_path", str(WEIGHTS_ROOT / "diffusion_pytorch_model.safetensors"),
        "--image_encoder_path", str(WEIGHTS_ROOT / "models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth"),
        "--pretrained_lora_path", str(WEIGHTS_ROOT / "EffectErase.ckpt"),
        "--num_inference_steps", str(num_inference_steps),
        "--height", str(height),
        "--width", str(width),
        "--num_frames", str(num_frames),
        "--frame_interval", str(frame_interval),
        "--seed", str(seed),
        "--cfg", str(cfg),
        "--lora_alpha", str(lora_alpha),
    ]
    if tiled:
        cmd.append("--tiled")
    if use_teacache:
        cmd.append("--use_teacache")

    JOBS[job_id] = {"status": "running"}
    threading.Thread(target=_run_job, args=(job_id, cmd, out_path, tmpdir), daemon=True).start()
    return JSONResponse({"job_id": job_id})

@app.get("/jobs/{job_id}")
async def job_status(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        return JSONResponse({"status": "failed", "error": "Not found"}, status_code=404)
    return JSONResponse({"status": job["status"], "error": job.get("error")})

@app.get("/jobs/{job_id}/result")
async def job_result(job_id: str):
    job = JOBS.get(job_id)
    if not job or job.get("status") != "done":
        return Response(status_code=404)
    data = job.pop("data")
    return Response(content=data, media_type="video/mp4")

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)


In [ ]:
%%bash
pkill -9 -f kaggle_effect_erase_server.py || true
pkill -9 -f cloudflared || true
sleep 2
nohup env EFFECT_ERASE_WEIGHTS_DIR=/kaggle/working/weights python /kaggle/working/kaggle_effect_erase_server.py > /kaggle/working/server.log 2>&1 &
nohup /usr/local/bin/cloudflared tunnel --url http://localhost:8000 > /kaggle/working/tunnel.log 2>&1 &
sleep 6
TUNNEL_URL=$(grep -o 'https://[-a-z0-9]*\.trycloudflare\.com' /kaggle/working/tunnel.log | head -n 1)
echo "========================================================================"
echo "Tunnel URL: $TUNNEL_URL"
echo "========================================================================"